# Understanding Recurrent Neural Networks (RNNs) and Backpropagation Through Time (BPTT)

Recurrent Neural Networks (RNNs) are designed to handle sequential data by maintaining a hidden state that captures information from previous inputs. They are particularly useful for tasks where context or sequential order is important, such as language modeling, time series forecasting, and sequence prediction.

---

# RNN Architecture

An RNN processes inputs one at a time while maintaining a hidden state that gets updated at each time step. The core equations governing the forward pass of an RNN are:

## 1) Hidden State Update

$h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$

## 2) Output Computation

$y_t = W_{hy}h_t + b_y$

### Where:

- $x_t$ is the input at time step $t$  
- $h_t$ is the hidden state at time step $t$  
- $W_{xh}$ is the weight matrix for input → hidden state  
- $W_{hh}$ is the weight matrix for hidden → hidden state  
- $W_{hy}$ is the weight matrix for hidden → output  
- $b_h$ and $b_y$ are bias terms  
- $\tanh$ is the hyperbolic tangent activation function applied element-wise  

---

# Forward Pass Implementation

In the forward pass, we iterate over each element in the input sequence, updating the hidden state and computing the output.

Steps:

1. Initialize the hidden state $h_0$ to zeros.
2. For each time step $t$:
   - Compute  
     $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$
   - Compute  
     $y_t = W_{hy}h_t + b_y$
   - Store $h_t$ and $y_t$ for use in backpropagation.

---

# Loss Function

The loss function measures the discrepancy between predicted outputs and actual targets.

For sequence prediction tasks we often use **Mean Squared Error (MSE)**:

$Loss = \frac{1}{T}\sum_{t=1}^{T}(\hat{y}_t - y_t)^2$

Where:

- $T$ is the sequence length  
- $\hat{y}_t$ is the predicted output  
- $y_t$ is the true target at time step $t$  

---

# Backpropagation Through Time (BPTT)

BPTT trains RNNs by **unrolling the network across time** and applying backpropagation through the sequence.

## Step 1 — Gradient of Loss w.r.t Output

$\frac{\partial L}{\partial y_t} = \hat{y}_t - y_t$

---

## Step 2 — Output Layer Gradients

$\frac{\partial L}{\partial W_{hy}} += \frac{\partial L}{\partial y_t} \cdot h_t^T$

$\frac{\partial L}{\partial b_y} += \frac{\partial L}{\partial y_t}$

---

## Step 3 — Backpropagate Through Hidden Layers

$\frac{\partial L}{\partial h_t} = W_{hy}^T \cdot \frac{\partial L}{\partial y_t} + dh_{next}$

$dh_{raw} = \frac{\partial L}{\partial h_t} \circ (1 - h_t^2)$

$dh_{next} = W_{hh}^T \cdot dh_{raw}$

Where:

- $\circ$ denotes **element-wise multiplication**
- $dh_{next}$ carries gradients from the next time step

Important:

- $dh_{next}$ must be initialized to **zero** at the final time step $T$.
- It is updated each iteration while moving backward in time.

---

## Step 4 — Hidden Layer Gradients

$\frac{\partial L}{\partial W_{xh}} += dh_{raw} \cdot x_t^T$

$\frac{\partial L}{\partial W_{hh}} += dh_{raw} \cdot h_{t-1}^T$

$\frac{\partial L}{\partial b_h} += dh_{raw}$

These steps are repeated **from time step $T$ down to $1$**, accumulating gradients across the sequence.

---

# Updating Weights

After computing gradients, we update parameters using gradient descent.

$W_{xh} -= \text{learning\_rate} \times \frac{\partial L}{\partial W_{xh}}$

$W_{hh} -= \text{learning\_rate} \times \frac{\partial L}{\partial W_{hh}}$

$W_{hy} -= \text{learning\_rate} \times \frac{\partial L}{\partial W_{hy}}$

$b_h -= \text{learning\_rate} \times \frac{\partial L}{\partial b_h}$

$b_y -= \text{learning\_rate} \times \frac{\partial L}{\partial b_y}$

---

# Implementing the RNN

To implement an RNN with BPTT:

### 1) Initialization
Initialize weights with small random values:

- $W_{xh}, W_{hh}, W_{hy}$ → small random values  
- $b_h, b_y$ → zeros  

---

### 2) Forward Pass

Process the input sequence:

- Update hidden states
- Compute outputs
- Store intermediate values for backpropagation

---

### 3) Backward Pass

Perform BPTT:

- Compute gradients backward through time
- Accumulate gradients
- Update parameters

---

### 4) Training Loop

Train for multiple epochs:

1. Forward pass  
2. Compute loss  
3. Backward pass (BPTT)  
4. Update parameters  

---

# Tips for Implementation

### Gradient Clipping
To prevent exploding gradients:
```python
clip(gradient, -threshold, threshold)
```


### Learning Rate
A high learning rate can cause unstable training.

### Debugging
Always verify matrix dimensions for:

- matrix multiplication
- broadcasting

### Testing
Start with **small hidden sizes and short sequences**.

---

# Example Calculation

Suppose we have:

Input sequence:

$x = [x_1, x_2]$

Target sequence:

$y = [y_1, y_2]$

---

## 1) Forward Pass

### Time step $t=1$

$h_1 = \tanh(W_{xh}x_1 + W_{hh}h_0 + b_h)$

$\hat{y}_1 = W_{hy}h_1 + b_y$

---

### Time step $t=2$

$h_2 = \tanh(W_{xh}x_2 + W_{hh}h_1 + b_h)$

$\hat{y}_2 = W_{hy}h_2 + b_y$

---

## 2) Compute Loss

$L = \frac{1}{2}\left[(\hat{y}_1 - y_1)^2 + (\hat{y}_2 - y_2)^2\right]$

---

## 3) Backward Pass

Starting from $t=2$ down to $t=1$.

### At $t=2$

$\frac{\partial L}{\partial \hat{y}_2} = \hat{y}_2 - y_2$

Backpropagate to compute gradients for:

- $W_{hy}$
- $b_y$
- $h_2$

---

### At $t=1$

Use the **chain rule** to propagate gradients further to:

- $W_{xh}$
- $W_{hh}$
- $b_h$

---

# Conclusion

RNNs trained with **Backpropagation Through Time (BPTT)** are powerful tools for modeling sequential data.

However, they suffer from:

- **Vanishing gradients**
- **Exploding gradients**

Solutions include:

- Gradient clipping
- **LSTM networks**
- **GRU networks**

Implementing an RNN from scratch is an excellent way to understand how modern sequence models work under the hood.

[Problem Desc](https://www.deep-ml.com/problems/62?from=Deep%20Learning)

In [7]:
import numpy as np
class SimpleRNN:
    def __init__(self, input_size, hidden_size, output_size):
        """
        Initializes the RNN with random weights and zero biases
        """
        
        self.hidden_size = hidden_size
        self.W_xh = np.random.randn(hidden_size, input_size)* 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size)* 0.01
        self.W_hy = np.random.randn(output_size, hidden_size)* 0.01
        self.b_h = np.zeros((hidden_size, 1))
        self.b_y = np.zeros((output_size, 1))
    
    def forward(self, x):
        """
        Forward pass through the RNN for a given sequence of inputs.
        """
        self.inputs = {}
        self.hidden_states = {-1: np.zeros((self.hidden_size, 1))}
        self.outputs = {}
        
        seq_length = x.shape[0]
        
        for t in range(seq_length):
            self.inputs[t] = x[t].reshape(-1, 1)
            
            self.hidden_states[t] = np.tanh(self.W_xh @ self.inputs[t] + self.W_hh @ self.hidden_states[t-1] + self.b_h)
            
            self.outputs[t] = self.W_hy @ self.hidden_states[t] + self.b_y
        
        return np.array(list(self.outputs.values())).reshape(seq_length, 1, -1)
    
    def backward(self, x, y, learning_rate):
        """
        Backpropagation through time to adjust weights based on error gradient.
        """
        seq_length = x.shape[0]
        
        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        dW_hy = np.zeros_like(self.W_hy)
        db_h = np.zeros_like(self.b_h)
        db_y = np.zeros_like(self.b_y)
        
        dh_next = np.zeros((self.hidden_size, 1))
        
        for t in reversed(range(seq_length)):
            dy = self.outputs[t] - y[t].reshape(-1, 1)
            
            dW_hy += dy @ self.hidden_states[t].T
            db_y += dy
            
            dh = self.W_hy.T @ dy + dh_next
            
            dtanh = (1 - self.hidden_states[t] ** 2) * dh
            
            dW_xh += dtanh @ self.inputs[t].T
            dW_hh += dtanh @ self.hidden_states[t-1].T
            db_h += dtanh
            
            dh_next = self.W_hh.T @ dtanh
            
        self.W_xh -= learning_rate * dW_xh
        self.W_hh -= learning_rate * dW_hh
        self.W_hy -= learning_rate * dW_hy
        self.b_h -= learning_rate * db_h
        self.b_y -= learning_rate * db_y

In [8]:
# Test Case 1
import numpy as np

np.random.seed(42) 

input_sequence = np.array([[1.0], [2.0], [3.0], [4.0]])
expected_output = np.array([[2.0], [3.0], [4.0], [5.0]])

rnn = SimpleRNN(input_size=1, hidden_size=5, output_size=1)

# Train the RNN over multiple epochs
for epoch in range(100):
    output = rnn.forward(input_sequence)
    rnn.backward(input_sequence, expected_output, learning_rate=0.01)

print(output)

# Expected Output:
# [[[2.24143915]],[[3.18450265]],[[4.04305928]],[[4.57419398]]]

[[[2.24143915]]

 [[3.18450265]]

 [[4.04305928]]

 [[4.57419398]]]


In [9]:
# Test Case 2
import numpy as np

np.random.seed(42) 

input_sequence = np.array([[1.0,2.0], [7.0,2.0], [1.0,3.0], [12.0,4.0]])
expected_output = np.array([[2.0], [3.0], [4.0], [5.0]])

rnn = SimpleRNN(input_size=2, hidden_size=3, output_size=1)

# Train the RNN over multiple epochs
for epoch in range(100):
    output = rnn.forward(input_sequence)
    rnn.backward(input_sequence, expected_output, learning_rate=0.01)

print(output)

# Expected Output:
# [[[2.42201379]],[[3.44167595]],[[3.6129965 ]],[[4.50660152]]]

[[[2.42201379]]

 [[3.44167595]]

 [[3.6129965 ]]

 [[4.50660152]]]
